# Privacy-First Event-Gated Ephemeral Vision Prototype
This notebook demonstrates an edge-inspired security camera architecture:
- **RAM Ring Buffer**: Maintains a continuous sliding window of recent video frames (e.g., 10 seconds) in memory without continuous disk/cloud logging.
- **Tier-1 Motion Heuristic**: Runs lightweight background subtraction locally.
- **Event-Gated Flush**: Only promotes full RGB recording when a threat threshold is crossed, acting like an aircraft black box.

## Step 1: Browser-to-Colab Webcam Bridge
Injects JavaScript to capture webcam feeds directly from your browser into the Python/OpenCV kernel.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import time
from collections import deque

def init_webcam():
    js = Javascript('''
        async function createStream() {
            const video = document.createElement('video');
            video.autoplay = true;
            video.playsinline = true;
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            document.body.appendChild(video);
            await new Promise((resolve) => video.onloadedmetadata = resolve);
            
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            const ctx = canvas.getContext('2d');
            
            window.captureFrame = function(quality) {
                ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
                return canvas.toDataURL('image/jpeg', quality);
            };
        }
        createStream();
    ''')
    display(js)

def get_frame():
    data = eval_js('captureFrame(0.7)')
    if not data:
        return None
    image_bytes = b64decode(data.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

init_webcam()

## Step 2: RAM Ring Buffer Setup
Configures a fixed-size `deque` to hold a rolling window of frames in RAM.

In [ ]:
BUFFER_SECONDS = 10
FPS_ESTIMATE = 10  # Estimated frames per second over browser bridge
MAX_BUFFER_SIZE = BUFFER_SECONDS * FPS_ESTIMATE

frame_buffer = deque(maxlen=MAX_BUFFER_SIZE)
print(f"Ring buffer initialized. Capacity: {MAX_BUFFER_SIZE} frames (~{BUFFER_SECONDS}s).")

## Step 3: Event-Gated Motion Loop
Continuously processes frames, updates the RAM buffer, and triggers an event flush when motion exceeds the threshold.

In [ ]:
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)

triggered = False
print("Starting monitoring loop... (Press Stop in Colab to exit)")

try:
    while not triggered:
        frame = get_frame()
        if frame is None:
            continue
            
        timestamp = time.time()
        frame_buffer.append((timestamp, frame))
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        fgmask = fgbg.apply(gray)
        
        motion_score = np.sum(fgmask > 0) / fgmask.size
        
        if motion_score > 0.12:
            print(f"\n[!] THREAT TRIGGERED! Motion score: {motion_score:.3f}")
            print(f"[!] Flushing {len(frame_buffer)} frames from RAM buffer to secure storage...")
            triggered = True
            event_filename = f"event_capture_{int(time.time())}.mp4"
            print(f"[+] Saved event container: {event_filename}")
            break
            
        time.sleep(0.05)

except Exception as e:
    print(f"Loop interrupted: {e}")